In [ ]:
from sklearn.linear_model import LogisticRegression

from FeDa4Fair.dataset import FairFederatedDataset
from FeDa4Fair.utils.data_utils import generate_multiobjective_bias
from FeDa4Fair.visualization import plot_multi_attribute_fairness

# Create a dataset with conflicting values for the same sensitive attribute

In this notebook we'll create a Federated Dataset in which we have two groups of clients. All of them are unfair toward the same sensitive attribute but in opposite directions.

We'll use the Dutch Dataset from HF, the sensitive attribute will be the gender, one group of clients will be unfair against male and one unfair toward female. 

In [ ]:

num_clients = 100
group_split = 50
client_names = [str(i) for i in range(num_clients)]

# Configure groups to show opposite bias directions for 'sex_binary'
# We use aggressive bias injection to ensure the model learns the bias.
group_configs = [
    {
        "group_id": "Group A (Favors 1)",
        "num_clients": group_split,
        "configs": [
            {
                "attribute": "sex_binary",
                "value": 0,  # Bias against 0 (favors 1)
                "drop_mean": 0.9, "drop_std": 0.05,
                "flip_mean": 0.3, "flip_std": 0.05
            }
        ]
    },
    {
        "group_id": "Group B (Favors 0)",
        "num_clients": num_clients - group_split,
        "configs": [
            {
                "attribute": "sex_binary",
                "value": 1,  # Bias against 1 (favors 0)
                "drop_mean": 0.3, "drop_std": 0.05,
                "flip_mean": 0.1, "flip_std": 0.05
            }
        ]
    }
]

modifications = generate_multiobjective_bias(num_clients, group_configs, client_names)

In [ ]:
# Initialize Dataset with Cross-Silo setting (train/test split per client)
fds = FairFederatedDataset(
    dataset="lucacorbucci/Dutch_census_binary_marital_status",
    split="all",
    partitioners={"train": num_clients},
    label_name="occupation_binary",
    sensitive_attributes=["sex_binary"],
    modification_dict=modifications,
    fl_setting="cross-silo",
    perc_train_val_test=[0.8, 0.2],
    client_names=client_names
)

print("Preparing dataset...")
fds.prepare()

In [ ]:
print("\nVerifying training sample counts per client:")
for client_id in range(num_clients):
partition = fds.load_partition(client_id, "train_train")
df = partition.to_pandas()
group_label = "Group A (Favors 1)" if client_id < 5 else "Group B (Favors 0)"
counts = df["sex_binary"].value_counts().to_dict()
print(f"Client {client_id:2} ({group_label}): 0 -> {counts.get(0, 0):4}, 1 -> {counts.get(1, 0):4}")


In [ ]:
print("\nGenerating model-based fairness plot...")

# We define colors: 0 -> Red (Favors 0), 1 -> Blue (Favors 1)
val_colors = {0: "red", 1: "blue"}

# Pass the model to plot_multi_attribute_fairness
# It will automatically train on 'train_train' and evaluate on 'train_test'
fig, ax, df_results = plot_multi_attribute_fairness(
    partitioner=fds.partitioners["train_train"],
    partitioner_test=fds.partitioners["train_test"],
    label_name="occupation_binary",
    sens_atts=["sex_binary"],
    fairness_metric="DP",
    model=LogisticRegression(max_iter=1000, solver="liblinear"),
    size_unit="value",
    value_colors=val_colors,
    fds=fds,
    split="train_train",
    test_split="train_test",
    title="Model Prediction Fairness (Red: favors 0, Blue: favors 1)"
)

In [ ]:
cols_to_print = ["Accuracy", "sex_binary_0_1", "sex_binary_1_0"]
print(df_results[cols_to_print])